<a href="https://colab.research.google.com/github/Rohan46os/50M-Model/blob/main/BoomLLm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import requests
import tiktoken

In [ ]:
Url = "https://raw.githubusercontent.com/karpathy/char-rnn/refs/heads/master/data/tinyshakespeare/input.txt"
data = requests.get(Url).text

In [ ]:
len(data)

1115394

In [ ]:
enc = tiktoken.get_encoding("o200k_base")
encoded_tokens = enc.encode(data)

In [ ]:
len(encoded_tokens)

297606

In [ ]:
batch_size = 4
sequence_length = 1024
batch_sequence = []

for i in range(batch_size):
  starting_index = i * sequence_length
  ending_index = sequence_length + starting_index
  batch_sequence.append(encoded_tokens[starting_index:ending_index])
  torched_sequence = torch.tensor(batch_sequence)

In [ ]:
torched_sequence.size()

torch.Size([4, 1024])

In [ ]:
vocab_size = 199997
d_embed = 64
theta = 1000

In [ ]:
Embedding_table_ = nn.Embedding(vocab_size, d_embed)
Embedded_table = Embedding_table_(torched_sequence)

In [ ]:
Embedded_table.size()

torch.Size([4, 1024, 64])

In [ ]:
Embedded_table.size()

torch.Size([4, 1024, 64])

In [ ]:
# Rope Embedding
x = Embedded_table #shape[4, 1024, 64]
# x = [] # word embedding
position_of_tokens = torch.arange(0, sequence_length).unsqueeze(1)#shape[1,1024]
frequencies = 1 / theta ** (2 * torch.arange(0, d_embed // 2) / d_embed)
cos_frequencies = torch.cos(frequencies*position_of_tokens).unsqueeze(0)
sin_frequencies = torch.sin(frequencies*position_of_tokens).unsqueeze(0)

# x is the input embedding of d= 64
x_0 = x[:, :, 0::2]
x_1 = x[:, :, 1::2]

Rope_x0 = x_0*cos_frequencies-x_1*sin_frequencies
Rope_x1 = x_0*sin_frequencies+x_1*cos_frequencies

stacked_pair = torch.stack([Rope_x0, Rope_x1], dim=-1)
final_pos_vec = stacked_pair.flatten(-2)

In [ ]:
stacked_pair.size()
final_pos_vec.size()

torch.Size([4, 1024, 64])

Completed writing a tokenizer(used tiktoken and) and divided the input into the size[B, N] and then embedded the token into a 64d and then passed the embedded token through the Rope

In [ ]:
#MHA
batch_size = 4
sequence_length = 1024
d_embed = 64
no_of_heads = 4
noh_d = d_embed//no_of_heads

Query = nn.Linear(d_embed, d_embed)
Key = nn.Linear(d_embed, d_embed)
Value = nn.Linear(d_embed, d_embed)
out_proj = nn.Linear(d_embed, d_embed)


q  = Query(final_pos_vec).reshape(batch_size, sequence_length, no_of_heads, noh_d).transpose(1,2)
k = Key(final_pos_vec).reshape(batch_size, sequence_length, no_of_heads, noh_d).transpose(1,2)
v = Value(final_pos_vec).reshape(batch_size, sequence_length, no_of_heads, noh_d).transpose(1,2)

#####Attention_Mechanism#######
attention_scores = ((q@k.transpose(-1, -2))/(noh_d)**0.5)#shape 4, 4, 1024, 1024]
####Masking(self attention)######
masking_matrix = torch.tril(torch.ones(sequence_length, sequence_length))
masked_attention_scores = attention_scores.masked_fill(masking_matrix ==0, float('-inf'))
########################################--Masked Attention--###############
sliding_window = 256
masking_matrix_1 = torch.tril(torch.ones(sequence_length, sequence_length), diagonal=0)
masking_matrix_2 = torch.tril(torch.ones(sequence_length, sequence_length), diagonal=-sliding_window)
final_mat = (masking_matrix_1-masking_matrix_2)
local_masked_attention_scores = attention_scores.masked_fill(final_mat ==0, float('-inf'))
soft_maxed_local_scores = torch.softmax(local_masked_attention_scores, dim=-1)
#############################-------#############################
soft_maxed_scores = torch.softmax(attention_scores, dim=-1)
final_attention_scores = (soft_maxed_scores@v).transpose(1, 2)#shape [4, 1024, 4, 16]
final_reshaped_score = final_attention_scores.reshape(batch_size, sequence_length, d_embed)
output_projection = out_proj(final_reshaped_score)

In [ ]:
#testing local attention
sliding_window = 2
masking_matrix_1 = torch.tril(torch.ones(10, 10), diagonal=0)
masking_matrix_2 = torch.tril(torch.ones(10, 10), diagonal=-sliding_window)
final_mat = (masking_matrix_1-masking_matrix_2)
final_mat

tensor([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 1., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 1., 1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 1., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 1., 1., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1., 1., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 1., 1.]])

In [ ]:
output_projection.size()

torch.Size([4, 1024, 64])

In [ ]:
########--------------------------Feed Forward Network-----------------------##########


feed_forward = nn.Sequential(
    nn.Linear(d_embed, 4*d_embed),
    nn.ReLU(),
    nn.Linear(4*d_embed, d_embed)
)
x_out = feed_forward(output_projection)


#####Final ouput head####  Required size(B, T, vocab_size) = 4, 1024, vocab_size
### Embeddding table size = vocab_size* 64

### 4, 1024, 199997 = (4, 1024, 64) * (199997, 64)
# 4, 1024, 64 * 64, 199997; so we can just transpose the Embedding table and multipy it withour output to get 4, 1024, 199997
#weight tieing for saving parameters


# Embedding_table_weights = Embedding_table_.weight
x_output = x_out@(Embedding_table_.weight).transpose(-1, -2) #[4, 1024, 199997]  here we used the Embedding_table_ weights as matrix to project the output in to vocab size to find probabilities

## to predict the next word we only need the last token so
logits = x_output[:, -1, :]

final_probs = torch.softmax(logits, dim=-1)


logit_selection = torch.multinomial(final_probs, num_samples=1) # We get the final token which is the index next predicted token




In [1]:
import torch
import torch.nn as nn
import requests
import tiktoken

Url = "https://raw.githubusercontent.com/karpathy/char-rnn/refs/heads/master/data/tinyshakespeare/input.txt"
data = requests.get(Url).text

enc = tiktoken.get_encoding("o200k_base")
encoded_tokens = enc.encode(data)

batch_size = 4
sequence_length = 1024
batch_sequence = []

batch_size = 4
sequence_length = 1024
d_embed = 64
no_of_heads = 4
noh_d = d_embed//no_of_heads

for i in range(batch_size):
  starting_index = i * sequence_length
  ending_index = sequence_length + starting_index
  batch_sequence.append(encoded_tokens[starting_index:ending_index])
torched_sequence = torch.tensor(batch_sequence)

vocab_size = 199997
d_embed = 64
theta = 1000

Embedding_table_ = nn.Embedding(vocab_size, d_embed)
Embedded_table = Embedding_table_(torched_sequence)

Query = nn.Linear(d_embed, d_embed)
Key = nn.Linear(d_embed, d_embed)
Value = nn.Linear(d_embed, d_embed)
out_proj = nn.Linear(d_embed, d_embed)


feed_forward = nn.Sequential(
    nn.Linear(d_embed, 4*d_embed),
    nn.ReLU(),
    nn.Linear(4*d_embed, d_embed)
)

###generation of next block
idx = torched_sequence

for _ in range(100):
  # Rope Embedding
  idx_cond = idx[:, -1024:]
  x = Embedding_table_(idx_cond) #shape[4, 1024, 64]
  # x = [] # word embedding
  position_of_tokens = torch.arange(0, sequence_length).unsqueeze(1)#shape[1,1024]
  frequencies = 1 / theta ** (2 * torch.arange(0, d_embed // 2) / d_embed)
  cos_frequencies = torch.cos(frequencies*position_of_tokens).unsqueeze(0)
  sin_frequencies = torch.sin(frequencies*position_of_tokens).unsqueeze(0)

  # x is the input embedding of d= 64
  x_0 = x[:, :, 0::2]
  x_1 = x[:, :, 1::2]

  Rope_x0 = x_0*cos_frequencies-x_1*sin_frequencies
  Rope_x1 = x_0*sin_frequencies+x_1*cos_frequencies

  stacked_pair = torch.stack([Rope_x0, Rope_x1], dim=-1)
  final_pos_vec = stacked_pair.flatten(-2)

  #MHA


  q  = Query(final_pos_vec).reshape(batch_size, sequence_length, no_of_heads, noh_d).transpose(1,2)
  k = Key(final_pos_vec).reshape(batch_size, sequence_length, no_of_heads, noh_d).transpose(1,2)
  v = Value(final_pos_vec).reshape(batch_size, sequence_length, no_of_heads, noh_d).transpose(1,2)

  #####Attention_Mechanism#######
  attention_scores = ((q@k.transpose(-1, -2))/(noh_d)**0.5)#shape 4, 4, 1024, 1024]
  ####Masking(self attention)######
  masking_matrix = torch.tril(torch.ones(sequence_length, sequence_length))
  masked_attention_scores = attention_scores.masked_fill(masking_matrix ==0, float('-inf'))

  soft_maxed_scores = torch.softmax(masked_attention_scores, dim=-1)
  final_attention_scores = (soft_maxed_scores@v).transpose(1, 2)#shape [4, 1024, 4, 16]
  final_reshaped_score = final_attention_scores.reshape(batch_size, sequence_length, d_embed)

  output_projection = out_proj(final_reshaped_score)

  x_out = feed_forward(output_projection)



  #####Final ouput head####  Required size(B, T, vocab_size) = 4, 1024, vocab_size
  ### Embeddding table size = vocab_size* 64

  ### 4, 1024, 199997 = (4, 1024, 64) * (199997, 64)
  # 4, 1024, 64 * 64, 199997; so we can just transpose the Embedding table and multipy it withour output to get 4, 1024, 199997
  #weight tieing for saving parameters


  # Embedding_table_weights = Embedding_table_.weight
  x_output = x_out@(Embedding_table_.weight).transpose(-1, -2) #[4, 1024, 199997]  here we used the Embedding_table_ weights as matrix to project the output in to vocab size to find probabilities

  ## to predict the next word we only need the last token so
  logits = x_output[:, -1, :]

  final_probs = torch.softmax(logits, dim=-1)


  logit_selection = torch.multinomial(final_probs, num_samples=1) # We get the final token which is the index next predicted token


  idx = torch.cat((idx, logit_selection), dim=1)



In [2]:
for i in range(batch_size):
    print(f"\n--- Sequence {i} ---")
    print(enc.decode(idx[i].tolist()))


--- Sequence 0 ---
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in t

In [ ]:
class ChatGpt(nn.Module):
  def __init__(self):
    super().__init__()
    self.Embedding_table = nn.Embedding(vocab_size, d_embed)


  def forward(self, torched_sequence):
    self.Embedded_table = self.Embedding_table(torched_sequence)
    return self.Embedded_table

  # def Rope_Encodding(self, X_input):